# CAWOT-CM coreset sweep (V0 + V1) — full-rigor protocol

**Plan-faithful sweep:**
- 4 methods (random / v0 / v0_proto / v1)
- 6 budgets {5, 10, 20, 30, 40, 50}% (full scaling curve, plan Part B2)
- 3 seeds (42, 1, 2) for error bars (plan Tuần 4-5)
- Per-category eval: **goal / full / wentwrong**, where `wentwrong = anomaly`
- Resumable (records.csv persisted)

**Total: 4 × 6 × 3 = 72 fine-tune runs ≈ 17 h.** Split across ~3 Kaggle sessions (~5.5 h each).
The runner is RESUMABLE — restart same notebook, it skips combos already in `records.csv`.

Setup: GPU **P100** + Internet **ON**.

## 1. Env + clone + install

In [ ]:
!nvidia-smi -L
import os
if not os.path.exists("/kaggle/working/cawot-cm"):
    !git clone https://github.com/HohoHocCode/cawot-cm.git /kaggle/working/cawot-cm
%cd /kaggle/working/cawot-cm
!pip install -q open_clip_torch faiss-gpu-cu12 einops huggingface_hub

## 2. Download 5 shards from HF

~7 GB download + ~14 GB extracted. Cached across sessions.

In [ ]:
!python scripts/setup_data.py --output /kaggle/working/pab_data --num-shards 5

## 3. Run the sweep (resumable)

- **Embeddings cached** after first call (~15 min once).
- **records.csv accumulates** — restart this cell resumes where it left off.
- Each fine-tune run ~3-23 min depending on budget; eval ~1.5 min.

**Multi-session strategy (recommended):** keep `train.seeds: [42, 1, 2]` in config. If Kaggle session times out mid-run, just rerun this cell — already-completed (method, budget, seed) combos are skipped. After ~3 sessions all 72 runs done.

Faster alternative: edit `config.yaml` to `train.seeds: [42]` for session 1, `[1]` for session 2, `[2]` for session 3. Equivalent result, more controlled per-session.

In [ ]:
!python scripts/run_sweep.py --config config.yaml

## 4. Results table (overall + per-category)

Shows mean ± std over completed seeds. If you've only run 1 seed so far, std is 0 (single value). After 3 seeds, std reflects training/selection noise.

In [ ]:
import json, pandas as pd
summary = json.load(open("/kaggle/working/outputs/eval/summary.json"))
print("zeroshot:", summary["zeroshot"])

methods = [m for m in ["random", "v0", "v0_proto", "v1"] if m in summary]
budgets = sorted({float(b) for m in methods for b in summary[m].keys()})
categories = ["overall", "goal", "full", "wentwrong"]

rows = []
for c in categories:
    for b in budgets:
        r = {"category": c, "budget": f"{int(b*100)}%"}
        for m in methods:
            s = summary[m].get(str(b), {}).get(c)
            r[m] = f"{s['mean_R@1_mean']:.2f}±{s['mean_R@1_std']:.2f}" if s else "-"
        rows.append(r)
df = pd.DataFrame(rows)
df

## 5. Plot R@1 vs budget — separate panels for overall + wentwrong (anomaly)

This is the key figure for the paper: does method ordering hold on the anomaly subset?

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
markers = {"random": "o", "v0": "^", "v0_proto": "v", "v1": "s"}
for ax, cat in zip(axes, ["overall", "wentwrong"]):
    x = [b * 100 for b in budgets]
    for m in methods:
        y = [summary[m].get(str(b), {}).get(cat, {}).get("mean_R@1_mean") for b in budgets]
        e = [summary[m].get(str(b), {}).get(cat, {}).get("mean_R@1_std", 0) for b in budgets]
        if all(v is None for v in y): continue
        ax.errorbar(x, y, yerr=e, marker=markers[m], capsize=3, label=m)
    zs = summary["zeroshot"].get(cat)
    if zs is not None:
        ax.axhline(zs, ls="--", c="gray", label=f"zero-shot ({zs:.1f})")
    ax.set_xlabel("Budget (% of train pool)"); ax.set_ylabel("mean R@1")
    ax.set_title(f"{cat}"); ax.legend(); ax.grid(alpha=0.3)
fig.suptitle("V0 family + V1 — Overall vs. Anomaly (wentwrong)")
plt.tight_layout()
plt.savefig("/kaggle/working/outputs/eval/sweep_curve.png", dpi=120, bbox_inches="tight")
plt.show()

## 6. How to read (key story for the report)

**Overall panel**: tổng hợp performance on i.i.d. val (mixed categories).
**Wentwrong panel**: anomaly subset — closest to the real PAB task.

Outcomes:
- If `v0_proto` wins overall AND wentwrong → prototype selection is the right call, paper can lean on it.
- If `v0_proto` wins overall but `v1` (or `v0`) wins wentwrong → **the gold story**: prototype helps typical, coverage/diversity needed for anomaly. Strong V1 justification.
- If methods near-tie at wentwrong → anomaly retrieval is hard for all → motivation for V2 (Wasserstein-aware budget) to specifically allocate more budget to anomaly-relevant clusters.

Artifacts to commit (Save Version → Save & Run All):
- `outputs/eval/records.csv`
- `outputs/eval/summary.json`
- `outputs/eval/sweep_curve.png`